In [13]:
import time
import os
import copy
import urllib.request
import shutil
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torchvision
from torchvision import datasets, transforms
from tensorflow.keras.applications import ResNet50
torch.manual_seed(42)
np.random.seed(42)

#Data
if not os.path.exists('inspiritai_util.py'):
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/inspirit-ai-data-bucket-1/Modules/inspiritai_util.py",
        "inspiritai_util.py",
    )

download_url = "https://storage.googleapis.com/inspirit-ai-data-bucket-1/Data/AI%20Scholars/Sessions%206%20-%2010%20(Projects)/Project%20-%20Towards%20Precision%20Medicine/"

data_dir = "data"
os.makedirs(data_dir, exist_ok=True)

for filename in ["images.npy", "labels.npy"]:
    file_path = os.path.join(data_dir, filename)
    if not os.path.exists(file_path):
        urllib.request.urlretrieve(download_url + filename, file_path)

images = np.load(os.path.join(data_dir, "images.npy"))
labels = np.load(os.path.join(data_dir, "labels.npy"))


# Pennylane
import pennylane as qml
from pennylane import numpy as pnp

# Plotting
import matplotlib.pyplot as plt

In [18]:
labels_ohe = np.array(pd.get_dummies(labels))
y = labels_ohe
X = images / 255.

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1)
print(X_train.shape)

standard_scaler = StandardScaler()
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

X_train_scaled = standard_scaler.fit_transform(X_train_flat)
X_test_scaled = standard_scaler.transform(X_test_flat)

pca = PCA(n_components=8)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

minmax_scaler = MinMaxScaler(feature_range=(-np.pi, np.pi))

X_train_final = minmax_scaler.fit_transform(X_train_pca)
X_test_final = minmax_scaler.transform(X_test_pca)
print(X_train_final.shape)

(768, 150, 150, 3)
(768, 8)


In [20]:
model = ResNet50(weights='imagenet', include_top=False, input_shape=(8, 8, 3))

n_qubits = 6
n_layers = 4
weights = pnp.random.uniform(0, np.pi, (n_layers, n_qubits, 3), requires_grad=True)
weights_shape = qml.StronglyEntanglingLayers.shape(n_layers=n_layers, n_wires=n_qubits)
x = np.random.uniform(0, np.pi, n_qubits)

dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def full_pipeline(x, weights):
    qml.AngleEmbedding(x, wires=range(n_qubits), rotation="Y")
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]
print(full_pipeline(x, weights))

def classifier(weights, bias, x):
    raw_outputs = full_pipeline(x, weights)
    stacked_outputs = qml.math.stack(raw_outputs)
    return qml.math.mean(stacked_outputs)

def square_loss(labels, predictions):
    preds = qml.math.stack(predictions)
    return qml.math.sum((labels - preds) ** 2) / len(labels)

def cost(weights, bias, X, y):
    preds = [classifier(weights, bias, x) for x in X]
    return square_loss(y, preds)

def accuracy(labels, weights, bias, X):
    preds = np.sign(np.array([float(classifier(weights, bias, x)) for x in X]))
    return np.mean(np.array(labels) == preds)

label = 1.0
prediction = 0.40

weights = pnp.random.uniform(0, pnp.pi, weights_shape, requires_grad=True)
bias = pnp.array(0.0, requires_grad=True)

X_sample = pnp.array(X_train_final[:5], requires_grad=False)
y_sample = pnp.array(y_train[:5], requires_grad=False)
print(cost(weights, bias, X_sample, y_sample))

ValueError: Input size must be at least 32x32; Received: input_shape=(8, 8, 3)